# GlucoCast — Glucose Prediction Demo

CGM trace loading → IOB/COB features → TFT predictions → Clarke Error Grid → alert generation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('imports ok')

In [ ]:
# --- synthetic CGM trace (5-min intervals, 24 hours) ---

INTERVAL_MIN = 5
N_STEPS = 24 * 60 // INTERVAL_MIN   # 288 readings

rng = np.random.default_rng(42)
t = np.arange(N_STEPS) * INTERVAL_MIN   # minutes from midnight

# base circadian rhythm
base = 95 + 15 * np.sin(2 * np.pi * t / (24 * 60) - np.pi / 2)

# simulate 3 meals (breakfast 7am, lunch 12pm, dinner 6pm)
def meal_response(t, peak_min, mag=70, decay=45):
    dt = t - peak_min
    resp = np.zeros_like(t, dtype=float)
    resp[dt >= 0] = mag * np.exp(-dt[dt >= 0] / decay)
    return resp

glucose = (
    base
    + meal_response(t, 7*60+20)
    + meal_response(t, 12*60+15, mag=55)
    + meal_response(t, 18*60+30, mag=80)
    + rng.normal(0, 4, N_STEPS)
)
glucose = np.clip(glucose, 55, 350)

timestamps = pd.date_range('2024-01-15 00:00', periods=N_STEPS, freq='5min')
cgm_df = pd.DataFrame({'timestamp': timestamps, 'glucose_mg_dl': glucose})

print(f'CGM trace: {len(cgm_df)} readings over 24 hours')
print(cgm_df.describe()['glucose_mg_dl'].round(1))

In [ ]:
# --- IOB and COB feature computation ---

def compute_iob(timestamps, insulin_events, dia_min=240):
    """
    Insulin on board using exponential decay model.
    insulin_events: list of (timestamp_min, dose_units)
    """
    iob = np.zeros(len(timestamps))
    t_min = np.array([(ts - timestamps[0]).total_seconds() / 60 for ts in timestamps])
    for event_t, dose in insulin_events:
        dt = t_min - event_t
        active = (dt >= 0) & (dt <= dia_min)
        iob[active] += dose * (1 - dt[active] / dia_min) ** 2
    return iob


def compute_cob(timestamps, carb_events, absorption_min=180):
    """
    Carbs on board using linear absorption model.
    """
    cob = np.zeros(len(timestamps))
    t_min = np.array([(ts - timestamps[0]).total_seconds() / 60 for ts in timestamps])
    for event_t, carbs_g in carb_events:
        dt = t_min - event_t
        active = (dt >= 0) & (dt <= absorption_min)
        cob[active] += carbs_g * (1 - dt[active] / absorption_min)
    return cob


# synthetic insulin and meal events
t0 = cgm_df['timestamp'].iloc[0]
insulin_events = [(420, 4.0), (720, 3.5), (1110, 5.0)]   # (min from midnight, units)
carb_events    = [(430, 45),  (730, 60),   (1120, 75)]    # (min, grams)

cgm_df['iob_units'] = compute_iob(cgm_df['timestamp'], insulin_events)
cgm_df['cob_grams'] = compute_cob(cgm_df['timestamp'], carb_events)
cgm_df['glucose_rate'] = cgm_df['glucose_mg_dl'].diff() / INTERVAL_MIN   # mg/dL/min

print('Features computed:')
print(cgm_df[['timestamp', 'glucose_mg_dl', 'iob_units', 'cob_grams', 'glucose_rate']]
      .iloc[80:86]
      .to_string(index=False))

In [ ]:
# --- TFT prediction: 30/60/120-min horizons with confidence intervals ---

def predict_glucose(
    cgm_series: np.ndarray,
    iob: np.ndarray,
    cob: np.ndarray,
    horizons_min: list = [30, 60, 120],
    interval: int = 5,
) -> dict:
    """
    Dummy TFT-style prediction.
    In production: wraps a trained pytorch-forecasting TFT model.
    """
    results = {}
    last_val = cgm_series[-1]
    last_rate = (cgm_series[-1] - cgm_series[-6]) / (5 * interval) if len(cgm_series) >= 6 else 0.0
    last_iob  = iob[-1]
    last_cob  = cob[-1]

    for h in horizons_min:
        steps = h // interval
        # naive linear trend + meal/insulin correction
        trend = last_val + last_rate * h
        meal_bump = 0.4 * last_cob * (1 - h / 180)
        insulin_drop = -12 * last_iob * (h / 60)
        pred_mean = float(np.clip(trend + meal_bump + insulin_drop, 50, 400))
        # uncertainty grows with horizon
        sigma = 8 + 0.15 * h
        results[h] = {
            'mean': round(pred_mean, 1),
            'lower_80': round(pred_mean - 1.28 * sigma, 1),
            'upper_80': round(pred_mean + 1.28 * sigma, 1),
            'lower_95': round(pred_mean - 1.96 * sigma, 1),
            'upper_95': round(pred_mean + 1.96 * sigma, 1),
        }
    return results


# predict from a point during the dinner rise (step 230 ≈ 19:10)
PRED_IDX = 230
preds = predict_glucose(
    cgm_series=cgm_df['glucose_mg_dl'].values[:PRED_IDX],
    iob=cgm_df['iob_units'].values[:PRED_IDX],
    cob=cgm_df['cob_grams'].values[:PRED_IDX],
)

print(f'Predictions from t={cgm_df["timestamp"].iloc[PRED_IDX].strftime("%H:%M")}')
print(f'Current glucose: {cgm_df["glucose_mg_dl"].iloc[PRED_IDX]:.0f} mg/dL')
print()
print(f'{"Horizon":<10} {"Mean":>7} {"80% CI":>18} {"95% CI":>22}')
print('-' * 60)
for h, v in preds.items():
    ci80 = f'[{v["lower_80"]}, {v["upper_80"]}]'
    ci95 = f'[{v["lower_95"]}, {v["upper_95"]}]'
    print(f'{h:>3} min    {v["mean"]:>7.1f} {ci80:>18} {ci95:>22}')

In [ ]:
# --- Clarke Error Grid ---

fig, ax = plt.subplots(figsize=(7, 7))

# generate synthetic reference vs predicted pairs
rng2 = np.random.default_rng(7)
ref  = rng2.uniform(60, 300, 120)
pred_noise = rng2.normal(0, 15, 120)
pred = np.clip(ref + pred_noise, 40, 400)

ax.scatter(ref, pred, s=14, alpha=0.6, color='steelblue', label='predictions (n=120)')
ax.plot([0, 400], [0, 400], 'k--', lw=0.8, label='perfect agreement')
ax.plot([0, 400], [0, 400*1.2], 'gray', lw=0.6, alpha=0.5)
ax.plot([0, 400], [0, 400*0.8], 'gray', lw=0.6, alpha=0.5)

# zone labels
for (tx, ty, label) in [(200, 195, 'Zone A'), (80, 200, 'Zone B'), (300, 80, 'Zone C')]:
    ax.text(tx, ty, label, fontsize=9, color='dimgray', ha='center')

ax.set_xlabel('Reference Glucose (mg/dL)', fontsize=11)
ax.set_ylabel('Predicted Glucose (mg/dL)', fontsize=11)
ax.set_title('Clarke Error Grid — 30-min Horizon', fontsize=12)
ax.set_xlim(0, 400); ax.set_ylim(0, 400)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/glucocast_ceg.png', dpi=120)
plt.show()
print('Clarke Error Grid saved to /tmp/glucocast_ceg.png')

In [ ]:
# --- alert generation ---

HYPO_THRESHOLD   = 70   # mg/dL
HYPER_THRESHOLD  = 180
RAPID_RISE_RATE  = 2.0  # mg/dL/min
RAPID_FALL_RATE  = -2.0

def generate_alerts(cgm_df: pd.DataFrame, predictions: dict):
    alerts = []
    current = cgm_df['glucose_mg_dl'].iloc[-1]
    rate    = cgm_df['glucose_rate'].iloc[-1]

    if current < HYPO_THRESHOLD:
        alerts.append({'severity': 'URGENT', 'type': 'hypoglycemia',
                        'message': f'Current glucose {current:.0f} mg/dL — BELOW 70'})
    if current > HYPER_THRESHOLD:
        alerts.append({'severity': 'WARNING', 'type': 'hyperglycemia',
                        'message': f'Current glucose {current:.0f} mg/dL — above target'})
    if rate >= RAPID_RISE_RATE:
        alerts.append({'severity': 'INFO', 'type': 'rapid_rise',
                        'message': f'Glucose rising at {rate:.1f} mg/dL/min ↑↑'})
    if rate <= RAPID_FALL_RATE:
        alerts.append({'severity': 'WARNING', 'type': 'rapid_fall',
                        'message': f'Glucose falling at {rate:.1f} mg/dL/min ↓↓'})

    # predictive alert
    if predictions.get(30, {}).get('mean', 999) < HYPO_THRESHOLD + 15:
        alerts.append({'severity': 'PREDICTIVE', 'type': 'predicted_low',
                        'message': f'30-min forecast: {predictions[30]["mean"]} mg/dL — risk of low'})
    return alerts


alerts = generate_alerts(cgm_df.iloc[:PRED_IDX], preds)
print(f'Active alerts at {cgm_df["timestamp"].iloc[PRED_IDX-1].strftime("%H:%M")}:')
if not alerts:
    print('  No alerts — glucose in range')
for a in alerts:
    print(f'  [{a["severity"]}] {a["message"]}')